#### Open Meteo API Guide
https://open-meteo.com/en/docs

Names of data columns available for hourly can be found on the above website.

To get the exact codename to specify models, select the models and look at the weblink (should have ?models=ABC).

In [0]:
# Set up and navigate to catalog.schema workspace
catalog = "open_meteo"
schema = "datasets"
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.catalog.setCurrentCatalog(catalog)
spark.catalog.setCurrentDatabase(schema)

In [0]:
# Create data table

# spark.sql("DROP TABLE IF EXISTS open_meteo.datasets.api_data")
spark.sql('''
CREATE TABLE IF NOT EXISTS open_meteo.datasets.api_data (
    country STRING COMMENT "Country name."
    ,model STRING COMMENT "Weather model code used for forecast."
    ,forecast_date DATE COMMENT "Date of weather forecast."
    ,forecast_hour INT COMMENT "Hour of weather forecast."
    ,retrieval_timestamp BIGINT COMMENT "Unix timestamp of when the data was retrieved."
    ,temperature_2m DOUBLE COMMENT "Air temperature at 2m above ground in Celsius."
    ,apparent_temperature DOUBLE COMMENT "Perceived air temperature by taking into account the effect of wind and humidity, in Celsius"
    ,relative_humidity_2m DOUBLE COMMENT "Relative humidity at 2m above ground in %."
    ,cloud_cover DOUBLE COMMENT "Total cloud cover as an area fraction"
    ,wind_speed_10m DOUBLE COMMENT "Wind speed at 10m above ground in m/s."
    ,rain DOUBLE COMMENT "Liquid precipitation in mm."
)
USING DELTA
CLUSTER BY AUTO
''')

DataFrame[]

In [0]:
# Open Meteo API configuration variables

import requests
import pandas as pd

latitude = 1.3521
longitude = 103.8198
hourly_data_columns = "temperature_2m,apparent_temperature,relative_humidity_2m,cloud_cover,wind_speed_10m,rain"
models = [
    'ecmwf_ifs' # European Centre for Medium-Range Weather Forecasts High Resolution Forecasts 9km
    ,'ecmwf_aifs025_single' # European Centre for Medium-Range Weather Forecasts AI Forecasting System 0.25degree resolution
    ,'jma_gsm' # Japan Meteorological Agency Global Spectral Model
    ,'gem_seamless' # Global Environment Multiscale, Canadian Weather Service
    ,'icon_global'
    ,'gfs_global'
]
past_days = 3 # 0 to 92
forecast_days = 0 # 0 to 16
timezone = "Asia/Singapore"

params = {
    "latitude": latitude
    ,"longitude": longitude
    ,"hourly": hourly_data_columns
    ,"models": models
    ,"past_days": past_days
    ,"forecast_days": forecast_days
    ,"timezone": timezone
}
url = "https://api.open-meteo.com/v1/forecast"

In [0]:
response = requests.get(url, params=params)
data = response.json()
hourly_data = data.get("hourly", {}) # get the hourly data, there can be other timescales
pdf = pd.DataFrame(hourly_data)
sdf = spark.createDataFrame(pdf)
print(sdf.printSchema())
display(sdf)open_meteo.datasets.api_data

root
 |-- time: string (nullable = true)
 |-- temperature_2m_ecmwf_ifs: double (nullable = true)
 |-- apparent_temperature_ecmwf_ifs: double (nullable = true)
 |-- relative_humidity_2m_ecmwf_ifs: long (nullable = true)
 |-- cloud_cover_ecmwf_ifs: long (nullable = true)
 |-- wind_speed_10m_ecmwf_ifs: double (nullable = true)
 |-- rain_ecmwf_ifs: double (nullable = true)
 |-- temperature_2m_ecmwf_aifs025_single: double (nullable = true)
 |-- apparent_temperature_ecmwf_aifs025_single: double (nullable = true)
 |-- relative_humidity_2m_ecmwf_aifs025_single: long (nullable = true)
 |-- cloud_cover_ecmwf_aifs025_single: long (nullable = true)
 |-- wind_speed_10m_ecmwf_aifs025_single: double (nullable = true)
 |-- rain_ecmwf_aifs025_single: double (nullable = true)
 |-- temperature_2m_jma_gsm: double (nullable = true)
 |-- apparent_temperature_jma_gsm: double (nullable = true)
 |-- relative_humidity_2m_jma_gsm: long (nullable = true)
 |-- cloud_cover_jma_gsm: long (nullable = true)
 |-- wind_

time,temperature_2m_ecmwf_ifs,apparent_temperature_ecmwf_ifs,relative_humidity_2m_ecmwf_ifs,cloud_cover_ecmwf_ifs,wind_speed_10m_ecmwf_ifs,rain_ecmwf_ifs,temperature_2m_ecmwf_aifs025_single,apparent_temperature_ecmwf_aifs025_single,relative_humidity_2m_ecmwf_aifs025_single,cloud_cover_ecmwf_aifs025_single,wind_speed_10m_ecmwf_aifs025_single,rain_ecmwf_aifs025_single,temperature_2m_jma_gsm,apparent_temperature_jma_gsm,relative_humidity_2m_jma_gsm,cloud_cover_jma_gsm,wind_speed_10m_jma_gsm,rain_jma_gsm,temperature_2m_gem_seamless,apparent_temperature_gem_seamless,relative_humidity_2m_gem_seamless,cloud_cover_gem_seamless,wind_speed_10m_gem_seamless,rain_gem_seamless,temperature_2m_icon_global,apparent_temperature_icon_global,relative_humidity_2m_icon_global,cloud_cover_icon_global,wind_speed_10m_icon_global,rain_icon_global,temperature_2m_gfs_global,apparent_temperature_gfs_global,relative_humidity_2m_gfs_global,cloud_cover_gfs_global,wind_speed_10m_gfs_global,rain_gfs_global
2026-06-27T00:00,26.2,31.4,88,5,6.0,0.0,26.5,31.9,86,0,5.0,0.0,26.0,32.8,97,54,1.3,0.1,27.4,33.0,87,0,8.3,0.0,27.4,33.8,88,34,2.8,0.0,27.7,32.0,79,21,11.0,0.0
2026-06-27T01:00,26.1,31.3,89,5,6.2,0.0,26.1,31.5,88,0,4.8,0.0,25.9,32.8,98,64,1.0,0.1,27.2,33.0,89,20,7.2,0.0,27.2,33.8,89,45,1.6,0.0,27.5,32.0,80,12,9.3,0.0
2026-06-27T02:00,25.3,30.9,92,3,2.7,0.0,25.8,31.2,89,3,4.4,0.0,25.8,32.7,98,71,0.8,0.1,26.9,32.9,90,24,6.1,0.0,26.8,33.3,90,33,2.3,0.0,27.3,31.8,81,10,9.7,0.0
2026-06-27T03:00,25.1,30.7,92,2,1.7,0.0,25.5,30.9,90,13,3.7,0.0,25.7,32.5,98,75,0.7,0.2,26.6,32.6,91,12,5.4,0.0,26.7,33.0,90,36,2.6,0.0,27.1,31.3,81,10,10.7,0.0
2026-06-27T04:00,25.0,30.7,93,13,1.5,0.0,25.2,30.7,91,27,2.7,0.0,25.6,32.5,99,77,0.8,0.2,26.4,32.4,92,8,5.4,0.0,26.6,32.9,90,35,2.4,0.0,27.1,31.2,79,10,9.7,0.0
2026-06-27T05:00,24.9,30.6,93,23,1.4,0.0,25.0,30.6,92,44,2.3,0.0,25.6,32.4,98,79,0.5,0.2,26.3,32.1,92,8,5.4,0.0,26.6,32.7,89,22,2.5,0.0,27.0,30.8,78,71,10.8,0.0
2026-06-27T06:00,24.5,30.2,96,77,2.1,0.0,24.9,30.6,93,61,1.9,0.0,25.6,32.4,98,79,0.5,0.2,26.2,31.9,92,8,6.1,0.0,26.3,32.2,89,12,2.8,0.0,27.0,30.9,78,66,10.1,0.0
2026-06-27T07:00,24.3,30.1,97,98,1.5,0.0,25.1,30.7,92,75,2.2,0.0,25.7,32.5,97,79,0.5,0.2,26.2,31.8,92,8,6.5,0.0,26.1,32.0,90,18,3.3,0.0,26.9,30.9,79,100,9.6,0.0
2026-06-27T08:00,25.2,31.5,98,100,1.9,0.0,25.4,31.1,91,84,2.2,0.0,26.0,32.7,95,79,0.4,0.2,26.8,32.7,88,8,4.0,0.0,26.7,33.0,90,44,3.2,0.0,27.6,32.4,80,100,8.0,0.0
2026-06-27T09:00,28.0,35.0,87,90,0.9,0.0,26.2,32.1,88,87,2.3,0.0,26.5,33.2,92,78,1.1,0.0,28.9,34.5,78,8,6.1,0.0,28.4,34.1,81,57,6.1,0.0,28.8,33.1,75,100,11.4,0.0
